# 🤖 Orion Knowledge Assistant
## Setup — Preparación del Volume con documentos

---

> **DATABRICKS · MOSAIC AI · SETUP**
> 
> Este notebook prepara el entorno para el laboratorio: crea el Volume en Unity Catalog y copia los documentos de Orion desde el workspace hacia ese Volume, dejándolos listos para ser consumidos por Agent Bricks.

| ⏱ ~5 minutos | ▶ Ejecutar antes del lab | ⚙ Serverless Compute v5 |
|:---:|:---:|:---:|

---

## 📋 Antes de ejecutar este notebook

Asegúrate de haber completado estos pasos:

1. **Descargaste o clonaste el repositorio** del curso en tu máquina local
2. **Importaste la carpeta completa** al workspace de Databricks:
   - En Databricks, ve a **Workspace** → tu carpeta de usuario
   - Haz clic en el ícono **+** → **Import**
   - Selecciona la carpeta del repo (que contiene este notebook y la carpeta `data/`)
3. **Verificas que la estructura en tu workspace se vea así:**

```
📁 tu-carpeta-en-workspace/
├── 📓 OKA_00_Setup_Volume.ipynb        ← este notebook
├── 📓 OKA_Lab_KnowledgeAssistant.ipynb ← el laboratorio
└── 📁 data/
    ├── orion_doc_1.pdf
    ├── orion_doc_2.pdf
    └── ... (documentos de Orion)
```

> ⚠️ **Importante:** La carpeta `data/` debe estar al mismo nivel que este notebook para que las rutas funcionen correctamente.

---

## ⚙️ Configuración

Ajusta las variables de esta celda si tu catalog o schema son diferentes a los valores por defecto.

> 💡 En el **Databricks Free Trial**, el catalog es `workspace` y el schema es `default`. Si no los cambiaste, no necesitas modificar nada aquí.

In [ ]:
# ─── CONFIGURACIÓN — ajusta si es necesario ───────────────────────────────
CATALOG = "workspace"   # Catalog en Unity Catalog
SCHEMA  = "default"     # Schema dentro del catalog
VOLUME  = "orion_docs"  # Nombre del Volume que se creará
# ──────────────────────────────────────────────────────────────────────────────

# Rutas derivadas (no modificar)
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print(f"📦 Catalog : {CATALOG}")
print(f"📂 Schema  : {SCHEMA}")
print(f"🗄️  Volume  : {VOLUME}")
print(f"📍 Ruta    : {volume_path}")

## Paso 1 — Detectar la ruta de los documentos en el workspace

Este paso detecta automáticamente dónde está la carpeta `data/` relativa a este notebook.

In [ ]:
import os

# Detectar la ruta de este notebook en el workspace
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir  = os.path.dirname(notebook_path)

# Ruta completa a la carpeta data/ en el workspace
workspace_data_path = f"/Workspace{notebook_dir}/data"

print(f"📓 Notebook en : /Workspace{notebook_dir}")
print(f"📁 Carpeta data: {workspace_data_path}")

# Verificar que existe
if os.path.exists(workspace_data_path):
    files = [f for f in os.listdir(workspace_data_path) if not f.startswith(".")]
    print(f"\n✅ Carpeta data/ encontrada con {len(files)} archivo(s):")
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(workspace_data_path, f)) // 1024
        print(f"   📄 {f}  ({size_kb} KB)")
else:
    print(f"\n❌ No se encontró la carpeta data/ en: {workspace_data_path}")
    print("   Verifica que importaste la carpeta completa del repositorio.")
    raise FileNotFoundError(f"Carpeta data/ no encontrada en {workspace_data_path}")

## Paso 2 — Crear el Volume en Unity Catalog

Si el Volume ya existe, se omite la creación. Si no existe, se crea automáticamente.

In [ ]:
# Verificar si el volume ya existe
existing = spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA} LIKE '{VOLUME}'").collect()

if existing:
    print(f"✅ El Volume '{VOLUME}' ya existe en {CATALOG}.{SCHEMA}")
    print(f"   Ruta: {volume_path}")
else:
    print(f"⏳ Creando Volume '{VOLUME}' en {CATALOG}.{SCHEMA} ...")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
    print(f"✅ Volume creado exitosamente.")
    print(f"   Ruta: {volume_path}")

## Paso 3 — Copiar documentos al Volume

Copia todos los archivos de la carpeta `data/` del workspace al Volume de Unity Catalog.

Los archivos ya existentes en el Volume se omiten para evitar sobreescrituras innecesarias.

In [ ]:
import shutil

files = [f for f in os.listdir(workspace_data_path) if not f.startswith(".")]

if not files:
    print("⚠️  No se encontraron archivos en la carpeta data/")
else:
    print(f"📋 Archivos a copiar: {len(files)}")
    print("-" * 50)

    copied   = 0
    skipped  = 0
    errors   = 0

    for filename in sorted(files):
        src  = os.path.join(workspace_data_path, filename)
        dest = os.path.join(volume_path, filename)

        if os.path.exists(dest):
            print(f"   ⏭️  {filename}  (ya existe, omitido)")
            skipped += 1
            continue

        try:
            shutil.copy2(src, dest)
            size_kb = os.path.getsize(dest) // 1024
            print(f"   ✅ {filename}  ({size_kb} KB)")
            copied += 1
        except Exception as e:
            print(f"   ❌ {filename}  — Error: {e}")
            errors += 1

    print("-" * 50)
    print(f"\n📊 Resumen:")
    print(f"   ✅ Copiados  : {copied}")
    print(f"   ⏭️  Omitidos  : {skipped}")
    print(f"   ❌ Errores   : {errors}")

## Paso 4 — Verificación final

Confirma que los archivos están correctamente disponibles en el Volume y listos para Agent Bricks.

In [ ]:
# Listar archivos en el Volume
volume_files = [f for f in os.listdir(volume_path) if not f.startswith(".")]

print(f"📦 Contenido del Volume: {volume_path}")
print("=" * 55)

total_kb = 0
for filename in sorted(volume_files):
    filepath = os.path.join(volume_path, filename)
    size_kb  = os.path.getsize(filepath) // 1024
    total_kb += size_kb
    ext = filename.split(".")[-1].upper()
    icon = "📄" if ext == "PDF" else "📝" if ext in ["DOCX","DOC"] else "📁"
    print(f"   {icon}  {filename:<45} {size_kb:>5} KB")

print("=" * 55)
print(f"   Total: {len(volume_files)} archivo(s)  |  {total_kb} KB")
print()

if volume_files:
    print("✅ Volume listo. Puedes continuar con el laboratorio.")
    print()
    print(f"   Cuando configures el agente en Agent Bricks,")
    print(f"   selecciona este Volume como fuente de conocimiento:")
    print()
    print(f"   📍 {CATALOG}.{SCHEMA}.{VOLUME}")
else:
    print("❌ El Volume está vacío. Revisa los pasos anteriores.")

---

## ✅ Setup completado

El entorno está listo para el laboratorio. Aquí un resumen de lo que se configuró:

| Elemento | Valor |
|----------|-------|
| **Catalog** | `workspace` *(o el que configuraste)* |
| **Schema** | `default` *(o el que configuraste)* |
| **Volume** | `orion_docs` |
| **Ruta del Volume** | `/Volumes/workspace/default/orion_docs` |

### ¿Qué sigue?

Abre el notebook **`OKA_Lab_KnowledgeAssistant_AgentBricks.ipynb`** y comienza el laboratorio.

Cuando en el **Paso A2** te pida seleccionar la fuente de conocimiento, elige:
- **Knowledge source type:** `Files in a Volume`
- **Source:** `workspace.default.orion_docs` *(o el catalog.schema.volume que configuraste)*

---

<div style="text-align:center; color:#8BA3BE; font-size:12px; margin-top:16px;">
  © 2026 · Setup notebook · Orion Knowledge Assistant · Agent Bricks Lab
</div>